Modified code from part 5 to test different sources sizes and shapes or to test advective flow. To run multiple cases and save results, it will be helpful to change the name of the generated folder `folder5` below.

In [ ]:
from smart import config, mesh, model, mesh_tools, visualization
from smart.units import unit
from smart.model_assembly import (
    Compartment,
    Parameter,
    Reaction,
    Species,
    SpeciesContainer,
    ParameterContainer,
    CompartmentContainer,
    ReactionContainer,
)
import logging
logger = logging.getLogger("smart")
import dolfin as d
from matplotlib import pyplot as plt
import pathlib
import numpy as np

folder5 = pathlib.Path("part5")
folder5.mkdir(exist_ok=True)

axisymm = True
AR = SET ASPECT RATIO HERE
sourceRad = SET SOURCE RADIUS HERE
if axisymm:
    z0 = 50.0
    dmesh, facet_markers, cell_markers = mesh_tools.create_2Dcell(outerExpr=f"(r/{z0})**2 + ((z-{z0})/{z0})**2 - 1",
                                        innerExpr=f"(r/({sourceRad}*{AR**(1/3)}))**2 + ((z-{z0})/({sourceRad}*{AR**(-2/3)}))**2 - 1",
                                        hEdge=5.0, hInnerEdge=0.05, outer_marker=10, interface_marker=11,
                                        outer_tag=1, half_cell=True)
else:
    z0 = 0.0
    dmesh, facet_markers, cell_markers = mesh_tools.create_multicell(cubeSize=50, locVec1=[[0,0,0]],
                                                  cellRad1=[sourceRad*AR**(1/3),sourceRad*AR**(1/3),sourceRad*AR**(-2/3)], 
                                                  hCube=5.0, hCell=0.5, 
                                                  interface_marker1=11, outer_marker=10,
                                                  extracell_tag=1)

mesh_tools.write_mesh(dmesh, facet_markers, cell_markers, str(folder5 / "extracell.h5"))


Now we define the compartments, where we can optionally introduce advection.

In [ ]:
D_unit = unit.um**2 / unit.s
conc_unit = unit.molecule / unit.um**3
surf_unit = unit.molecule / unit.um**2
if axisymm:
    EC_var = Compartment("EC", 2, unit.um, 1, vel=[SET VELOCITY HERE (ex: [0.0,0.0,0.0] or ["10*y","0","0"])])
    source_var = Compartment("source", 1, unit.um, 11)
    outer_var = Compartment("outer", 1, unit.um, 10)
else:
    EC_var = Compartment("EC", 3, unit.um, 1, vel=[SET VELOCITY HERE (ex: [0.0,0.0,0.0] or ["10*y","0","0"])])
    source_var = Compartment("source", 2, unit.um, 11)
    outer_var = Compartment("outer", 2, unit.um, 10)
c_var = Species("c", 0, conc_unit, 130.0, D_unit, "EC")
kdecay_var = Parameter("kdecay", 0.011, 1/unit.s)
r_decay = Reaction("r_decay", ["c"], [], param_map={"k":"kdecay"},
                    species_map={"c":"c"}, eqn_f_str="k*c")
j0_var = Parameter("j0", 100.0, surf_unit/unit.s)
r_release = Reaction("r_release", [], ["c"],
                     param_map={"j0":"j0"}, eqn_f_str="j0",
                     explicit_restriction_to_domain="source")
# create containers
cc = CompartmentContainer()
cc.add([EC_var, source_var])
sc = SpeciesContainer()
sc.add([c_var])
pc = ParameterContainer()
pc.add([kdecay_var, j0_var])
rc = ReactionContainer()
rc.add([r_decay, r_release])
parent_mesh = mesh.ParentMesh(mesh_filename=str(folder5 / "extracell.h5"), 
                              mesh_filetype="hdf5", name="parent_mesh")
config_cur = config.Config()
model_cur = model.Model(pc, sc, cc, rc, config_cur, parent_mesh)
config_cur.solver.update({"final_t": 1.0, "initial_dt": 0.05})
config_cur.flags.update({"axisymmetric_model": axisymm})
model_cur.initialize()
results = dict()
for species_name, species in model_cur.sc.items:
    results[species_name] = d.XDMFFile(model_cur.mpi_comm_world, 
                                       str(folder5 / f"{species_name}.xdmf"))
    results[species_name].parameters["flush_output"] = True
    results[species_name].write(model_cur.sc[species_name].sol, model_cur.t)

# An ROI can be defined to integrate over a specific subvolume.
# If use_roi is false, then the integral for c_roi is conducted over the source surface
use_roi = False
if use_roi:
    class ROI(d.SubDomain):
        def inside(self, x, on_boundary):
            rcur = np.sqrt(x[0]**2 + x[1]**2 + (x[2]-z0)**2)
            return (rcur > 5.5) and (rcur < 6.5)
    if axisymm:
        mf_int = d.MeshFunction("size_t", model_cur.cc["EC"].dolfin_mesh, 2, 0)
    else:
        mf_int = d.MeshFunction("size_t", model_cur.cc["EC"].dolfin_mesh, 3, 0)
    roiDef = ROI()
    roiDef.mark(mf_int, 1) # mark roi with 1
    dx = d.Measure("dx", domain=model_cur.cc["EC"].dolfin_mesh, subdomain_data=mf_int)
else:
    dx = d.Measure("dx", domain=model_cur.cc["EC"].dolfin_mesh)
    dx_source = d.Measure("dx", domain=model_cur.cc["source"].dolfin_mesh)
if axisymm:
    xEC = d.SpatialCoordinate(model_cur.cc["EC"].dolfin_mesh)
    volume = d.assemble_mixed(xEC[0]*dx)
    avg_c = [d.assemble_mixed(model_cur.sc["c"].sol*xEC[0]*dx)/volume]
    if use_roi:
        roi_vol = d.assemble_mixed(xEC[0]*dx(1))
        c_roi = [d.assemble_mixed(model_cur.sc["c"].sol*xEC[0]*dx(1))/roi_vol]
    else:
        xSource = d.SpatialCoordinate(model_cur.cc["source"].dolfin_mesh)
        roi_vol = d.assemble_mixed(xSource[0]*dx_source)
        c_roi = [d.assemble_mixed(model_cur.sc["c"].sol*xSource[0]*dx_source)/roi_vol]
else:
    volume = d.assemble_mixed(1.0*dx)
    avg_c = [d.assemble_mixed(model_cur.sc["c"].sol*dx)/volume]
    if use_roi:
        roi_vol = d.assemble_mixed(1.0*dx(1))
        c_roi = [d.assemble_mixed(model_cur.sc["c"].sol*dx(1))/roi_vol]
    else:
        roi_vol = d.assemble_mixed(1.0*dx_source)
        c_roi = [d.assemble_mixed(model_cur.sc["c"].sol*dx_source)/roi_vol]
logger.setLevel(logging.WARNING) # suppress excessive output
while True:
    model_cur.monolithic_solve()
    print(f"Done with t = {model_cur.t}")
    for species_name, species in model_cur.sc.items:
        results[species_name].write(model_cur.sc[species_name].sol, model_cur.t)
    if axisymm:
        avg_c.append(d.assemble_mixed(model_cur.sc["c"].sol*xEC[0]*dx) / volume)
        if use_roi:
            c_roi.append(d.assemble_mixed(model_cur.sc["c"].sol*xEC[0]*dx(1))/roi_vol)
        else:
            c_roi.append(d.assemble_mixed(model_cur.sc["c"].sol*xSource[0]*dx_source)/roi_vol)
    else:
        avg_c.append(d.assemble_mixed(model_cur.sc["c"].sol*dx) / volume)
        if use_roi:
            c_roi.append(d.assemble_mixed(model_cur.sc["c"].sol*dx(1))/roi_vol)
        else:
            c_roi.append(d.assemble_mixed(model_cur.sc["c"].sol*dx_source)/roi_vol)
    if model_cur.t >= model_cur.final_t:
        break

# save data for later plotting
tvec = [float(val) for val in model_cur.tvec]
np.savetxt(str(folder5 / "c_roi.txt"), np.column_stack((tvec, c_roi)))
np.savetxt(str(folder5 / "avg_c.txt"), np.column_stack((tvec, avg_c)))

Now plot the results and compare to the analytical solution.

In [ ]:
import numpy as np
from scipy.special import erfc
# now plot and compare to analytical solution
plt.plot(model_cur.tvec, c_roi, label="SMART")
# analytical solution at source membrane
t = np.array([float(val) for val in model_cur.tvec])
t[0] = 1e-6
j0 = j0_var.value
R = 5.0
dr = 0.0
D = c_var.D
k = kdecay_var.value
multFactor = j0*R**2/(dr + R)
term1 = (1/(2*(D+R*np.sqrt(D*k))))*np.exp(-dr/np.sqrt(D/k))*erfc(dr/(2*np.sqrt(D*t)) - np.sqrt(k*t))
term2 = (1/(2*(D-R*np.sqrt(D*k))))*np.exp(dr/np.sqrt(D/k))*erfc(dr/(2*np.sqrt(D*t)) + np.sqrt(k*t))
term3 = (-1/(D-k*R**2))*np.exp(dr/R + (D/R**2 - k)*t)*erfc(dr/(2*np.sqrt(D*t)) + np.sqrt(D*t)/R)
cAnalytical = multFactor*(term1 + term2 + term3)
plt.plot(t, cAnalytical, label="Analytical")

plt.xlabel('Time (s)')
plt.ylabel('c concentration $\mathrm{(molecules/μm^3)}$')
plt.legend()
plt.savefig(str(folder5 / "part5plot.png"))
plt.show()

visualization.plot(model_cur.sc["c"].sol)